# Experimento Formal de Clustering para Representaciones Textuales

## Objetivo general
Comparar 4 variantes de clustering implementadas en `validation/clustering.py` sobre 3 representaciones textuales (BoW, TF-IDF y embeddings Nomic) y 3 datasets (DialogSum, StackOverflow, ESQAD).

## Preguntas de investigación
1. ¿Qué combinación algoritmo-distancia logra mejor equilibrio entre cohesión y separación (ASW, CH)?
2. ¿Cuándo las métricas difusas (PC, PE, XB) aportan evidencia adicional respecto al clustering duro?
3. ¿Cómo cambia el rendimiento al pasar de representaciones dispersas (BoW/TF-IDF) a embeddings densos (Nomic)?

## Hipótesis iniciales
1. Las variantes con distancia coseno serán competitivas en datos textuales.
2. FCM capturará mejor ambiguedad temática en datasets heterogéneos.
3. Los embeddings Nomic tenderán a mayor separabilidad semántica, con mayor coste computacional.

## Diseño metodológico

Diseño factorial completo:
- 4 algoritmos: KMeans-euclidean, KMeans-cosine, FCM-euclidean, FCM-cosine
- 3 representaciones: BoW, TF-IDF, Nomic embeddings
- 3 datasets: DialogSum, StackOverflow, ESQAD
- N semillas por condición

Total de corridas = 4 x 3 x 3 x |K| x |semillas|.

Se incluye modo piloto para validar pipeline extremo a extremo antes de lanzar el barrido completo.

In [1]:
# Imports y configuración global
from __future__ import annotations

import importlib.util
import json
import logging
import os
import pickle
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

# Asegura que el root del repo este en sys.path, incluso si el notebook se ejecuta desde validation/notebooks
CANDIDATE_ROOT = Path.cwd().resolve()
if CANDIDATE_ROOT.name == "notebooks" and CANDIDATE_ROOT.parent.name == "validation":
    ROOT = CANDIDATE_ROOT.parent.parent
else:
    ROOT = next(
        (p for p in [CANDIDATE_ROOT, *CANDIDATE_ROOT.parents] if (p / "validation" / "clustering.py").exists()),
        CANDIDATE_ROOT,
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def _load_local_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module

# Carga explícita de módulos locales para evitar colisiones con paquetes externos llamados "validation"
clustering_module = _load_local_module("tfg_validation_clustering", ROOT / "validation" / "clustering.py")
datasets_module = _load_local_module("tfg_validation_datasets", ROOT / "validation" / "datasets.py")
metrics_module = _load_local_module("tfg_validation_metrics", ROOT / "validation" / "metrics" / "metrics.py")
emb_module = _load_local_module("tfg_validation_ollama_embeddings", ROOT / "validation" / "representation" / "ollama_embeddings.py")

GenericKMeans = clustering_module.GenericKMeans
DatasetLoader = datasets_module.DatasetLoader
evaluate_fuzzy_clustering = metrics_module.evaluate_fuzzy_clustering
evaluate_hard_clustering = metrics_module.evaluate_hard_clustering
OllamaEmbeddings = emb_module.OllamaEmbeddings

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print(f"ROOT detectado: {ROOT}")
print(f"UMAP disponible: {HAS_UMAP}")

/home/gabriel/clase/TFG/TFG-Chatbot/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT detectado: /home/gabriel/clase/TFG/TFG-Chatbot
UMAP disponible: True


In [2]:
# Configuración de logging y rutas de salida
RESULTS_DIR = ROOT / "validation" / "results" / "clustering_experiment"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
ARTIFACTS_DIR = RESULTS_DIR / "artifacts"
LOGS_DIR = RESULTS_DIR / "logs"

for p in [RESULTS_DIR, TABLES_DIR, FIGURES_DIR, ARTIFACTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

log_file = LOGS_DIR / "experiment.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.FileHandler(log_file), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("clustering_experiment")

def save_table(df: pd.DataFrame, filename: str) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    return path

def save_figure(fig: plt.Figure, filename: str, dpi: int = 200) -> Path:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path

print(f"Resultados en: {RESULTS_DIR}")

Resultados en: /home/gabriel/clase/TFG/TFG-Chatbot/validation/results/clustering_experiment


In [3]:
# Carga de datasets
loader = DatasetLoader()

# Ajusta límites si deseas acelerar en fase piloto
PILOT_MODE = True
PILOT_LIMITS = {
    "dialogsum": 1200,
    "stackoverflow": 1200,
    "esquad": 1200,
}

df_dialogsum = loader.load_dialogsum()
df_stack = loader.load_stackoverflow(limit=PILOT_LIMITS["stackoverflow"] if PILOT_MODE else 5000)
df_esquad = loader.load_esquad()

if PILOT_MODE:
    df_dialogsum = df_dialogsum.head(PILOT_LIMITS["dialogsum"])
    df_esquad = df_esquad.head(PILOT_LIMITS["esquad"])

for df, name in [
    (df_dialogsum, "dialogsum"),
    (df_stack, "stackoverflow"),
    (df_esquad, "esquad"),
]:
    df["dataset"] = name

datasets_raw = {
    "dialogsum": df_dialogsum[["text", "label", "dataset"]].copy(),
    "stackoverflow": df_stack[["text", "label", "dataset"]].copy(),
    "esquad": df_esquad[["text", "label", "dataset"]].copy(),
}

{k: v.shape for k, v in datasets_raw.items()}

Loading DialogSum...
Loading Stack Overflow subset...
Loading ESQAD (Spanish)...


{'dialogsum': (1200, 3), 'stackoverflow': (1200, 3), 'esquad': (1190, 3)}

In [4]:
# Limpieza y control de calidad
MIN_TEXT_LEN = 15

def clean_dataset(df: pd.DataFrame, min_len: int = 15) -> pd.DataFrame:
    out = df.copy()
    out = out.dropna(subset=["text"])
    out["text"] = out["text"].astype(str).str.strip()
    out = out[out["text"].str.len() > 0]
    out = out.drop_duplicates(subset=["text"])
    out = out[out["text"].str.len() >= min_len]
    return out.reset_index(drop=True)

datasets = {name: clean_dataset(df, min_len=MIN_TEXT_LEN) for name, df in datasets_raw.items()}
{k: v.shape for k, v in datasets.items()}

{'dialogsum': (1200, 3), 'stackoverflow': (1125, 3), 'esquad': (1184, 3)}

## Reporte descriptivo de muestra

En la siguiente celda se reportan, por dataset:
1. Número de documentos
2. Longitud media del texto
3. Número de etiquetas reales observadas

In [5]:
report_rows = []
for name, df in datasets.items():
    report_rows.append({
        "dataset": name,
        "n_docs": len(df),
        "avg_text_len": float(df["text"].str.len().mean()),
        "n_unique_labels": int(df["label"].nunique()),
    })

df_sample_report = pd.DataFrame(report_rows).sort_values("dataset")
save_table(df_sample_report, "sample_report.csv")
df_sample_report

,dataset,n_docs,avg_text_len,n_unique_labels
0,dialogsum,1200,740.635833,1006
2,esquad,1184,68.238176,1
1,stackoverflow,1125,199.427556,1


In [6]:
# Funciones de representación
def build_bow(texts: list[str], params: dict) -> np.ndarray:
    start = time.perf_counter()
    vectorizer = CountVectorizer(**params)
    X = vectorizer.fit_transform(texts)
    X = X.astype(np.float64)
    elapsed = time.perf_counter() - start
    return X, vectorizer, elapsed

def build_tfidf(texts: list[str], params: dict) -> np.ndarray:
    start = time.perf_counter()
    vectorizer = TfidfVectorizer(**params)
    X = vectorizer.fit_transform(texts)
    X = X.astype(np.float64)
    elapsed = time.perf_counter() - start
    return X, vectorizer, elapsed

def build_nomic_embeddings(texts: list[str], batch_size: int, model_name: str, host: str = "localhost", port: int = 11434):
    start = time.perf_counter()
    embedder = OllamaEmbeddings(model=model_name, host=host, port=port, timeout=60.0)
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        Xb = embedder.embed_batch(batch, normalize=True)
        vectors.append(Xb)
    X = np.vstack(vectors) if vectors else np.zeros((len(texts), 768), dtype=np.float32)
    elapsed = time.perf_counter() - start
    return X, embedder, elapsed

In [7]:
# Configuración por representación
REP_CONFIG = {
    "bow": {
        "max_features": 3000 if not PILOT_MODE else 1200,
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
    },
    "tfidf": {
        "max_features": 3000 if not PILOT_MODE else 1200,
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True,
    },
    "nomic": {
        "model_name": "nomic-embed-text",
        "batch_size": 64 if PILOT_MODE else 128,
        "host": os.getenv("OLLAMA_HOST", "localhost"),
        "port": int(os.getenv("OLLAMA_PORT", "11434")),
    },
}
REP_CONFIG

{'bow': {'max_features': 1200,
  'ngram_range': (1, 2),
  'min_df': 2,
  'max_df': 0.95},
 'tfidf': {'max_features': 1200,
  'ngram_range': (1, 2),
  'min_df': 2,
  'max_df': 0.95,
  'sublinear_tf': True},
 'nomic': {'model_name': 'nomic-embed-text',
  'batch_size': 64,
  'host': 'localhost',
  'port': 11434}}

In [8]:
# Construcción efectiva y cache de representaciones
representations: dict[str, dict[str, np.ndarray]] = {}
representation_meta: list[dict] = []

for ds_name, df in datasets.items():
    texts = df["text"].tolist()
    representations[ds_name] = {}

    # BoW
    bow_cache = ARTIFACTS_DIR / f"{ds_name}_bow.pkl"
    if bow_cache.exists():
        with open(bow_cache, "rb") as f:
            payload = pickle.load(f)
        X_bow = payload["X"]
        bow_elapsed = payload.get("elapsed_sec", np.nan)
    else:
        X_bow, bow_vec, bow_elapsed = build_bow(texts, REP_CONFIG["bow"])
        X_bow = normalize(X_bow, norm="l2", axis=1)
        with open(bow_cache, "wb") as f:
            pickle.dump({"X": X_bow, "vectorizer": bow_vec, "elapsed_sec": bow_elapsed}, f)
    representations[ds_name]["bow"] = X_bow
    representation_meta.append({"dataset": ds_name, "representation": "bow", "shape": str(X_bow.shape), "build_sec": bow_elapsed})

    # TF-IDF
    tfidf_cache = ARTIFACTS_DIR / f"{ds_name}_tfidf.pkl"
    if tfidf_cache.exists():
        with open(tfidf_cache, "rb") as f:
            payload = pickle.load(f)
        X_tfidf = payload["X"]
        tfidf_elapsed = payload.get("elapsed_sec", np.nan)
    else:
        X_tfidf, tfidf_vec, tfidf_elapsed = build_tfidf(texts, REP_CONFIG["tfidf"])
        X_tfidf = normalize(X_tfidf, norm="l2", axis=1)
        with open(tfidf_cache, "wb") as f:
            pickle.dump({"X": X_tfidf, "vectorizer": tfidf_vec, "elapsed_sec": tfidf_elapsed}, f)
    representations[ds_name]["tfidf"] = X_tfidf
    representation_meta.append({"dataset": ds_name, "representation": "tfidf", "shape": str(X_tfidf.shape), "build_sec": tfidf_elapsed})

    # Nomic
    nomic_cache = ARTIFACTS_DIR / f"{ds_name}_nomic.npy"
    nomic_meta = ARTIFACTS_DIR / f"{ds_name}_nomic_meta.json"
    if nomic_cache.exists() and nomic_meta.exists():
        X_nomic = np.load(nomic_cache)
        meta = json.loads(nomic_meta.read_text())
        nomic_elapsed = meta.get("elapsed_sec", np.nan)
    else:
        try:
            X_nomic, embedder, nomic_elapsed = build_nomic_embeddings(
                texts=texts,
                batch_size=REP_CONFIG["nomic"]["batch_size"],
                model_name=REP_CONFIG["nomic"]["model_name"],
                host=REP_CONFIG["nomic"]["host"],
                port=REP_CONFIG["nomic"]["port"],
            )
            X_nomic = normalize(X_nomic, norm="l2", axis=1)
            np.save(nomic_cache, X_nomic)
            nomic_meta.write_text(json.dumps({"elapsed_sec": nomic_elapsed}))
        except Exception as e:
            logger.warning(f"No se pudieron generar embeddings Nomic para {ds_name}: {e}")
            X_nomic = None
            nomic_elapsed = np.nan
    representations[ds_name]["nomic"] = X_nomic
    representation_meta.append({"dataset": ds_name, "representation": "nomic", "shape": str(None if X_nomic is None else X_nomic.shape), "build_sec": nomic_elapsed})

df_rep_meta = pd.DataFrame(representation_meta)
save_table(df_rep_meta, "representation_build_report.csv")
df_rep_meta

2026-04-14 20:58:27,812 | INFO | tfg_validation_ollama_embeddings | Initialized Ollama embeddings: http://localhost:11434/api/embeddings
2026-04-14 20:58:28,448 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
2026-04-14 20:58:28,672 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
2026-04-14 20:58:28,839 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
2026-04-14 20:58:28,990 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
2026-04-14 20:58:29,135 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
2026-04-14 20:58:29,231 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
2026-04-14 20:58:29,312 | INFO | httpx | HTTP Request: POST http://localhost:11434/api/embeddings "HTTP/1.1 200 OK"
2026-04-14 20:58:29,664 | INFO | httpx | HTTP Reque

,dataset,representation,shape,build_sec
0,dialogsum,bow,"(1200, 1200)",0.168985
1,dialogsum,tfidf,"(1200, 1200)",0.172550
2,dialogsum,nomic,"(1200, 768)",241.525717
3,stackoverflow,bow,"(1125, 1200)",0.033507
4,stackoverflow,tfidf,"(1125, 1200)",0.031609
5,stackoverflow,nomic,"(1125, 768)",71.163992
6,esquad,bow,"(1184, 1200)",0.018171
7,esquad,tfidf,"(1184, 1200)",0.016087
8,esquad,nomic,"(1184, 768)",44.607220


## Política de hiperparámetros

Parámetros base:
- $k \in [2, 15]$ (acotado por tamaño del dataset)
- semillas: 3 en piloto, 5 en ejecución completa
- `max_iter = 200`, `tol = 1e-4`
- FCM con `m = 2.0`

## Criterio de comparación exploratoria

Criterio principal:
- `ASW` (mayor es mejor)

Criterios de apoyo:
- `CH` (mayor es mejor)
- `runtime_sec` (menor es mejor)
- `n_iter` (menor es mejor)
- Para FCM: `XB` (menor), `PC` (mayor), `PE` (menor)

Regla de desempate para selección final:
1. Mayor `asw_mean`
2. Menor `runtime_sec_mean`
3. Menor `n_iter_mean`

In [9]:
# Grid experimental
K_VALUES = list(range(2, 8)) if PILOT_MODE else list(range(2, 16))
SEEDS = [42, 52, 62] if PILOT_MODE else [42, 52, 62, 72, 82]

ALGO_CONFIGS = [
    {"algorithm": "kmeans", "distance": "euclidean", "m": None},
    {"algorithm": "kmeans", "distance": "cosine", "m": None},
    {"algorithm": "fcm", "distance": "euclidean", "m": 2.0},
    {"algorithm": "fcm", "distance": "cosine", "m": 2.0},
]

rows = []
for ds_name, rep_dict in representations.items():
    for rep_name, X in rep_dict.items():
        if X is None:
            continue
        n = X.shape[0]
        k_values_ds = [k for k in K_VALUES if k < n]
        for cfg in ALGO_CONFIGS:
            for k in k_values_ds:
                for seed in SEEDS:
                    rows.append({
                        "dataset": ds_name,
                        "representation": rep_name,
                        "algorithm": cfg["algorithm"],
                        "distance": cfg["distance"],
                        "m": cfg["m"],
                        "k": k,
                        "seed": seed,
                    })

df_grid = pd.DataFrame(rows)
save_table(df_grid, "experiment_grid.csv")
df_grid.head(), len(df_grid)

(     dataset representation algorithm   distance   m  k  seed
 0  dialogsum            bow    kmeans  euclidean NaN  2    42
 1  dialogsum            bow    kmeans  euclidean NaN  2    52
 2  dialogsum            bow    kmeans  euclidean NaN  2    62
 3  dialogsum            bow    kmeans  euclidean NaN  3    42
 4  dialogsum            bow    kmeans  euclidean NaN  3    52,
 648)

In [10]:
# Runner principal
def _to_dense_if_needed(X):
    return X.toarray() if hasattr(X, "toarray") else X

def run_single_experiment(
    X,
    dataset_name: str,
    representation_name: str,
    algorithm: str,
    distance: str,
    k: int,
    seed: int,
    m: float | None = None,
    max_iter: int = 200,
    tol: float = 1e-4,
) -> dict:
    X_arr = _to_dense_if_needed(X)

    model = GenericKMeans(
        n_clusters=k,
        algorithm=algorithm,
        distance=distance,
        max_iter=max_iter,
        tol=tol,
        random_state=seed,
        m=2.0 if m is None else m,
    )

    start = time.perf_counter()
    model.fit(X_arr)
    elapsed = time.perf_counter() - start

    if algorithm == "fcm":
        metrics = evaluate_fuzzy_clustering(
            X_arr,
            model.membership_,
            model.centroids_,
            m=2.0 if m is None else m,
            distance=distance,
        )
    else:
        metrics = evaluate_hard_clustering(X_arr, model.labels_, metric=distance)

    return {
        "dataset": dataset_name,
        "representation": representation_name,
        "algorithm": algorithm,
        "distance": distance,
        "m": m,
        "k": k,
        "seed": seed,
        "asw": metrics.asw,
        "ch": metrics.ch,
        "pc": metrics.pc,
        "pe": metrics.pe,
        "xb": metrics.xb,
        "inertia": model.inertia_,
        "n_iter": model.n_iter_,
        "runtime_sec": elapsed,
    }

In [ ]:
# Bucle completo del experimento
results = []
errors = []

for _, row in df_grid.iterrows():
    ds = row["dataset"]
    rep = row["representation"]
    X = representations[ds][rep]

    try:
        out = run_single_experiment(
            X=X,
            dataset_name=ds,
            representation_name=rep,
            algorithm=row["algorithm"],
            distance=row["distance"],
            k=int(row["k"]),
            seed=int(row["seed"]),
            m=row["m"],
        )
        results.append(out)
    except Exception as e:
        errors.append({
            "dataset": ds,
            "representation": rep,
            "algorithm": row["algorithm"],
            "distance": row["distance"],
            "k": int(row["k"]),
            "seed": int(row["seed"]),
            "error": str(e),
        })

df_results = pd.DataFrame(results)
df_errors = pd.DataFrame(errors)

if not df_results.empty:
    save_table(df_results, "raw_results.csv")
if not df_errors.empty:
    save_table(df_errors, "execution_errors.csv")

df_results.head(), len(df_results), len(df_errors)

2026-04-14 21:05:20,609 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=euclidean
2026-04-14 21:05:20,857 | INFO | tfg_validation_clustering | K-Means converged at iteration 14
2026-04-14 21:05:20,859 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=euclidean, inertia=814.4248


In [ ]:
# Agregación por condición (media, desviación estándar, IC95)
def ci95(series: pd.Series) -> float:
    s = series.dropna()
    if len(s) <= 1:
        return np.nan
    return 1.96 * float(s.std(ddof=1)) / np.sqrt(len(s))

group_cols = ["dataset", "representation", "algorithm", "distance", "k"]
metric_cols = ["asw", "ch", "pc", "pe", "xb", "runtime_sec", "n_iter"]

agg_dict = {}
for mcol in metric_cols:
    agg_dict[f"{mcol}_mean"] = (mcol, "mean")
    agg_dict[f"{mcol}_std"] = (mcol, "std")
    agg_dict[f"{mcol}_ci95"] = (mcol, ci95)

df_summary = df_results.groupby(group_cols).agg(**agg_dict).reset_index()
save_table(df_summary, "summary_by_condition.csv")
df_summary.head()

In [ ]:
# Ranking y selección
# Criterio principal: ASW alto, con desempate por runtime y n_iter
df_rank = df_summary.copy()

df_rank["rank_asw"] = df_rank.groupby(["dataset", "representation"])["asw_mean"].rank(ascending=False, method="min")
df_rank["rank_runtime"] = df_rank.groupby(["dataset", "representation"])["runtime_sec_mean"].rank(ascending=True, method="min")
df_rank["rank_n_iter"] = df_rank.groupby(["dataset", "representation"])["n_iter_mean"].rank(ascending=True, method="min")

df_rank = df_rank.sort_values([
    "dataset", "representation", "asw_mean", "runtime_sec_mean", "n_iter_mean"
], ascending=[True, True, False, True, True])

df_best_per_condition = (
    df_rank
    .groupby(["dataset", "representation"], as_index=False, sort=False)
    .first()
)

df_best_global = df_rank.sort_values(["asw_mean", "runtime_sec_mean", "n_iter_mean"], ascending=[False, True, True]).head(20)

save_table(df_rank, "summary_ranked_full.csv")
save_table(df_best_per_condition, "summary_best_per_condition.csv")
save_table(df_best_global, "summary_best_global_top20.csv")

df_best_per_condition

In [ ]:
# Resumen exploratorio global (sin inferencia estadística)
df_summary_explore = df_summary.copy()
df_summary_explore["variant"] = df_summary_explore["algorithm"] + "_" + df_summary_explore["distance"]

df_global_by_variant = (
    df_summary_explore
    .groupby(["algorithm", "distance", "representation"], as_index=False)
    .agg(
        asw_mean=("asw_mean", "mean"),
        ch_mean=("ch_mean", "mean"),
        runtime_sec_mean=("runtime_sec_mean", "mean"),
        n_iter_mean=("n_iter_mean", "mean"),
        xb_mean=("xb_mean", "mean"),
        pc_mean=("pc_mean", "mean"),
        pe_mean=("pe_mean", "mean"),
    )
)

df_top_configs_by_asw = (
    df_summary_explore
    .sort_values(["asw_mean", "runtime_sec_mean", "n_iter_mean"], ascending=[False, True, True])
    .head(20)
    .reset_index(drop=True)
)

save_table(df_best_per_condition, "summary_best_per_condition.csv")
save_table(df_global_by_variant, "summary_global_by_variant.csv")
save_table(df_top_configs_by_asw, "summary_top_configs_by_asw.csv")

display(df_best_per_condition.head(10))
display(df_global_by_variant.sort_values(["asw_mean", "runtime_sec_mean"], ascending=[False, True]).head(10))
df_top_configs_by_asw.head(10)

In [ ]:
# Curvas de métrica vs k
curve_metrics = ["asw_mean", "ch_mean", "xb_mean", "pc_mean", "pe_mean"]

for (ds, rep), grp in df_summary.groupby(["dataset", "representation"]):
    fig, axes = plt.subplots(1, len(curve_metrics), figsize=(5 * len(curve_metrics), 4), sharex=True)
    for ax, metric in zip(axes, curve_metrics):
        plot_df = grp.copy()
        plot_df["variant"] = plot_df["algorithm"] + "_" + plot_df["distance"]
        sns.lineplot(data=plot_df, x="k", y=metric, hue="variant", marker="o", ax=ax)
        ax.set_title(f"{metric} vs k")
        ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"Metricas vs k | {ds} | {rep}", y=1.02)
    save_figure(fig, f"curves_{ds}_{rep}.png")
    plt.close(fig)

print("Curvas guardadas.")

In [ ]:
# Boxplots de estabilidad entre semillas
df_box = df_results.copy()
df_box["variant"] = df_box["algorithm"] + "_" + df_box["distance"]

for (ds, rep), grp in df_box.groupby(["dataset", "representation"]):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(data=grp, x="variant", y="asw", ax=axes[0])
    axes[0].set_title(f"ASW estabilidad | {ds} | {rep}")
    axes[0].tick_params(axis="x", rotation=30)

    sns.boxplot(data=grp, x="variant", y="ch", ax=axes[1])
    axes[1].set_title(f"CH estabilidad | {ds} | {rep}")
    axes[1].tick_params(axis="x", rotation=30)

    save_figure(fig, f"boxplot_stability_{ds}_{rep}.png")
    plt.close(fig)

print("Boxplots guardados.")

# Heatmaps por dataset: calidad (ASW) y coste (runtime)
for ds, grp_ds in df_summary.groupby("dataset"):
    tmp = grp_ds.copy()
    tmp["algorithm_distance"] = tmp["algorithm"] + "_" + tmp["distance"]

    asw_heat = tmp.groupby(["algorithm_distance", "representation"], as_index=False)["asw_mean"].mean()
    asw_pivot = asw_heat.pivot(index="algorithm_distance", columns="representation", values="asw_mean")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.heatmap(asw_pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"ASW medio por variante | {ds}")
    save_figure(fig, f"heatmap_asw_{ds}.png")
    plt.close(fig)

    runtime_heat = tmp.groupby(["algorithm_distance", "representation"], as_index=False)["runtime_sec_mean"].mean()
    runtime_pivot = runtime_heat.pivot(index="algorithm_distance", columns="representation", values="runtime_sec_mean")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.heatmap(runtime_pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
    ax.set_title(f"Runtime medio (s) por variante | {ds}")
    save_figure(fig, f"heatmap_runtime_{ds}.png")
    plt.close(fig)

# Scatter global calidad-coste
df_scatter = df_summary.copy()
df_scatter["algorithm_distance"] = df_scatter["algorithm"] + "_" + df_scatter["distance"]
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=df_scatter,
    x="runtime_sec_mean",
    y="asw_mean",
    hue="representation",
    style="algorithm_distance",
    s=90,
    alpha=0.85,
    ax=ax,
)
ax.set_title("Trade-off global calidad-coste (ASW vs runtime)")
ax.set_xlabel("Runtime medio (s)")
ax.set_ylabel("ASW medio")
ax.legend(loc="best", fontsize=8)
save_figure(fig, "scatter_quality_vs_cost.png")
plt.close(fig)

print("Heatmaps y scatter calidad-coste guardados.")

In [ ]:
# UMAP 2D (o PCA fallback)
from sklearn.decomposition import PCA

def reduce_to_2d(X, method: str = "umap", random_state: int = 42):
    X_arr = X.toarray() if hasattr(X, "toarray") else X
    if method == "umap" and HAS_UMAP:
        reducer = umap.UMAP(n_components=2, random_state=random_state, metric="cosine")
        return reducer.fit_transform(X_arr)
    pca = PCA(n_components=2, random_state=random_state)
    return pca.fit_transform(X_arr)

df_best = df_best_per_condition.copy()

for _, row in df_best.iterrows():
    ds, rep = row["dataset"], row["representation"]
    X = representations[ds][rep]
    if X is None:
        continue

    model = GenericKMeans(
        n_clusters=int(row["k"]),
        algorithm=row["algorithm"],
        distance=row["distance"],
        random_state=RANDOM_SEED,
        m=2.0 if pd.isna(row["m"]) else float(row["m"]),
    )
    X_arr = X.toarray() if hasattr(X, "toarray") else X
    labels_pred = model.fit_predict(X_arr)

    X2 = reduce_to_2d(X, method="umap", random_state=RANDOM_SEED)
    labels_true = datasets[ds]["label"].astype(str).values

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(X2[:, 0], X2[:, 1], c=labels_pred, s=10, cmap="tab20")
    axes[0].set_title(f"UMAP/PCA por cluster predicho | {ds}-{rep}")

    # Mapea etiquetas reales a enteros para colorear
    _, true_ids = np.unique(labels_true, return_inverse=True)
    axes[1].scatter(X2[:, 0], X2[:, 1], c=true_ids, s=10, cmap="tab20")
    axes[1].set_title(f"UMAP/PCA por etiqueta real | {ds}-{rep}")

    save_figure(fig, f"umap_{ds}_{rep}.png")
    plt.close(fig)

print("Figuras UMAP/PCA guardadas.")

In [ ]:
# t-SNE 2D para contraste cualitativo
for _, row in df_best_per_condition.iterrows():
    ds, rep = row["dataset"], row["representation"]
    X = representations[ds][rep]
    if X is None:
        continue

    X_arr = X.toarray() if hasattr(X, "toarray") else X
    n_samples = X_arr.shape[0]
    if n_samples < 50:
        continue

    model = GenericKMeans(
        n_clusters=int(row["k"]),
        algorithm=row["algorithm"],
        distance=row["distance"],
        random_state=RANDOM_SEED,
        m=2.0 if pd.isna(row["m"]) else float(row["m"]),
    )
    labels_pred = model.fit_predict(X_arr)

    perplexity = min(30, max(5, n_samples // 50))
    tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=perplexity, init="pca")
    X2 = tsne.fit_transform(X_arr)

    labels_true = datasets[ds]["label"].astype(str).values
    _, true_ids = np.unique(labels_true, return_inverse=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(X2[:, 0], X2[:, 1], c=labels_pred, s=10, cmap="tab20")
    axes[0].set_title(f"t-SNE por cluster predicho | {ds}-{rep}")

    axes[1].scatter(X2[:, 0], X2[:, 1], c=true_ids, s=10, cmap="tab20")
    axes[1].set_title(f"t-SNE por etiqueta real | {ds}-{rep}")

    save_figure(fig, f"tsne_{ds}_{rep}.png")
    plt.close(fig)

print("Figuras t-SNE guardadas.")

In [ ]:
# Incertidumbre difusa (solo FCM)
fuzzy_uncertainty_rows = []

for _, row in df_best_per_condition.iterrows():
    if row["algorithm"] != "fcm":
        continue

    ds, rep = row["dataset"], row["representation"]
    X = representations[ds][rep]
    if X is None:
        continue

    X_arr = X.toarray() if hasattr(X, "toarray") else X
    model = GenericKMeans(
        n_clusters=int(row["k"]),
        algorithm="fcm",
        distance=row["distance"],
        random_state=RANDOM_SEED,
        m=2.0 if pd.isna(row["m"]) else float(row["m"]),
    )
    model.fit(X_arr)
    U = model.membership_

    eps = 1e-12
    entropy = -np.sum(U * np.log(np.maximum(U, eps)), axis=1)
    max_membership = U.max(axis=1)

    fuzzy_uncertainty_rows.append({
        "dataset": ds,
        "representation": rep,
        "entropy_mean": float(np.mean(entropy)),
        "entropy_std": float(np.std(entropy)),
        "max_membership_mean": float(np.mean(max_membership)),
        "max_membership_std": float(np.std(max_membership)),
    })

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(entropy, bins=30, color="steelblue", alpha=0.8)
    axes[0].set_title(f"Entropia de membresia | {ds}-{rep}")

    axes[1].hist(max_membership, bins=30, color="darkorange", alpha=0.8)
    axes[1].set_title(f"Max membership | {ds}-{rep}")

    save_figure(fig, f"fuzzy_uncertainty_{ds}_{rep}.png")
    plt.close(fig)

df_fuzzy_uncertainty = pd.DataFrame(fuzzy_uncertainty_rows)
if not df_fuzzy_uncertainty.empty:
    save_table(df_fuzzy_uncertainty, "fuzzy_uncertainty_summary.csv")

df_fuzzy_uncertainty

## Discusión guiada

Completar tras ejecutar:
1. Algoritmo ganador por dataset y representación (según ASW y desempate por coste).
2. Consistencia de tendencias entre semillas y valores de k.
3. Trade-off calidad-coste entre variantes (ASW/CH frente a runtime y n_iter).
4. Diferencias por representación y por dataset (BoW, TF-IDF, Nomic).

Sugerencia: usar `summary_best_per_condition.csv`, `summary_global_by_variant.csv` y `summary_top_configs_by_asw.csv` como base del texto final.

Nota metodológica: en esta iteración no se realizan pruebas de significancia; las conclusiones son descriptivas y exploratorias.

## Amenazas a la validez

1. Sensibilidad a hiperparámetros (`k`, `m`, `max_features`, `min_df`, `perplexity`).
2. Sesgo por tamaño muestral en modo piloto y posibles truncamientos.
3. UMAP/t-SNE son técnicas estocásticas y dependientes de parámetros.
4. Coste computacional y disponibilidad de Nomic/Ollama pueden alterar comparabilidad práctica.
5. Etiquetas reales en algunos datasets pueden ser gruesas o parciales para validación externa.
6. No se realizan inferencias estadísticas en esta versión; los hallazgos deben interpretarse como evidencia descriptiva, no confirmatoria.

## Conclusiones y trabajo futuro

Plantilla para cierre:
1. Resumen de mejor configuración por dataset.
2. Balance rendimiento-calidad (ASW/CH/XB vs tiempo).
3. Recomendaciones metodológicas para el sistema TFG-Chatbot.
4. Siguientes pasos: HDBSCAN, robustez cross-domain, validación externa adicional (ARI/NMI si hay etiquetas consistentes), análisis de sensibilidad completo.

Nota operativa: tras validar piloto, desactivar `PILOT_MODE` para ejecución final completa sin alterar metodología.